# Drone Tespit ve Takibi — YOLO11 + ByteTrack

Bu notebook projenin tamamını Google Colab'da baştan sona çalıştırır:
1. Veri setini indirme ve hazırlama
2. YOLO11n eğitimi
3. Test setinde değerlendirme (boyuta göre recall dahil)
4. Videoda ByteTrack ile takip
5. ONNX'e çevirme ve CPU hız ölçümü

**Başlamadan önce:** Çalışma Zamanı → Çalışma zamanı türünü değiştir → **T4 GPU** seç.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/semanur-s/drone-detection-tracking.git
%cd drone-detection-tracking
!pip install -q -r requirements.txt

## (İsteğe bağlı) Sonuçları Google Drive'a kaydet
Colab oturumu kapanınca dosyalar silinir. Drive'ı bağlarsan eğitim çıktıları kaybolmaz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/drone-detection-tracking'

## 1. Veri setini hazırla
Seraphim Drone Detection Dataset (CC BY 4.0) zip'lenmiş batch'ler halinde tutuluyor. Tamamı ~9 GB olduğu için ilk birkaç batch'i indiriyoruz. Eğitimden %10 doğrulama seti ayrılır, test seti ayrı kalır.

In [ ]:
!python src/prepare_data.py --train-batches 3 --test-batches 1 --out data

In [ ]:
# Birkaç örnek görüntüyü etiketleriyle gör
import cv2, random, matplotlib.pyplot as plt
from pathlib import Path
imgs = sorted(Path('data/drone/train/images').glob('*.jpg'))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.ravel(), random.Random(0).sample(imgs, 8)):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    lbl = p.parent.parent / 'labels' / (p.stem + '.txt')
    for line in (lbl.read_text().splitlines() if lbl.exists() else []):
        _, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
        cv2.rectangle(im, (x1, y1), (int(x1 + bw*w), int(y1 + bh*h)), (255, 0, 0), 2)
    ax.imshow(im); ax.axis('off')
plt.tight_layout(); plt.savefig('assets/ornek_veriler.png', dpi=100); plt.show()

## 2. Eğitim
COCO'da önceden eğitilmiş YOLO11n ağırlıklarıyla başlayıp drone sınıfı için ince ayar yapıyoruz. T4 GPU'da veri miktarına göre yaklaşık 30–60 dakika sürer.

In [ ]:
!python src/train.py --data data/drone.yaml --model yolo11n.pt --epochs 50 --imgsz 640 --batch 16 --device 0
WEIGHTS = open('results/best_weights.txt').read().strip()
print(WEIGHTS)

In [ ]:
# Eğitim eğrileri (kayıp, precision, recall, mAP)
from IPython.display import Image, display
run_dir = Path(WEIGHTS).parent.parent
display(Image(str(run_dir / 'results.png'), width=1000))
import shutil; shutil.copy(run_dir / 'results.png', 'assets/egitim_egrileri.png')

## 3. Test setinde değerlendirme
Toplam mAP'in yanında küçük / orta / büyük drone'lar için ayrı recall hesaplanır. Uzaktaki drone'lar çok küçük göründüğü için asıl zorluk küçük grupta.

In [ ]:
!python src/evaluate.py --weights $WEIGHTS --data data/drone.yaml

In [ ]:
# Test tahminlerinden örnekler
from ultralytics import YOLO
model = YOLO(WEIGHTS)
test_imgs = sorted(Path('data/drone/test/images').glob('*.jpg'))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.ravel(), random.Random(1).sample(test_imgs, 8)):
    r = model.predict(str(p), conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.tight_layout(); plt.savefig('assets/test_tahminleri.png', dpi=100); plt.show()
shutil.copy(Path('runs/detect/test_eval/confusion_matrix.png'), 'assets/confusion_matrix.png') if Path('runs/detect/test_eval/confusion_matrix.png').exists() else None
shutil.copy(Path('runs/detect/test_eval/BoxPR_curve.png'), 'assets/pr_egrisi.png') if Path('runs/detect/test_eval/BoxPR_curve.png').exists() else None

## 4. Videoda takip (ByteTrack)
Bir drone videosu yükle. Ücretsiz lisanslı drone videoları için Pexels'te "drone flying sky" araması yapabilirsin. Gökyüzünde uçan bir drone'un **göründüğü** videolar seçilmeli, drone'dan çekilmiş videolar değil.

In [ ]:
from google.colab import files
up = files.upload()
VIDEO = list(up.keys())[0]
print(VIDEO)

In [ ]:
!python src/track.py --weights $WEIGHTS --source "$VIDEO" --tracker configs/bytetrack_drone.yaml

In [ ]:
# Takip videosundan bir kare kaydet ve videoyu indir
cap = cv2.VideoCapture('results/tracked.mp4'); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2); ok, frame = cap.read(); cap.release()
cv2.imwrite('assets/takip_karesi.jpg', frame)
plt.figure(figsize=(12, 7)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
files.download('results/tracked.mp4')

## 5. ONNX'e çevirme ve CPU hız ölçümü
Gömülü cihazlarda (Raspberry Pi, Jetson) genelde ONNX gibi optimize çalışma zamanları kullanılır. Burada CPU'da PyTorch ile ONNX Runtime'ı karşılaştırıyoruz.

In [ ]:
!python src/export_benchmark.py --weights $WEIGHTS --images data/drone/test/images --n 100

## 6. Sonuçları topla
Aşağıdaki çıktıyı ve `assets/` klasöründeki görselleri README'ye eklemek için kullan.

In [ ]:
import json
for f in ['test_metrics.json', 'track_stats.json', 'benchmark.json']:
    p = Path('results') / f
    if p.exists():
        print(f'--- {f}'); print(p.read_text())
!zip -qr sonuclar.zip results assets
files.download('sonuclar.zip')

In [ ]:
# Drive'a yedekle (1. hücrede Drive'ı bağladıysan)
import shutil, os
if 'SAVE_DIR' in globals():
    os.makedirs(SAVE_DIR, exist_ok=True)
    for d in ['results', 'assets', str(run_dir)]:
        shutil.copytree(d, os.path.join(SAVE_DIR, Path(d).name), dirs_exist_ok=True)
    print('Kaydedildi:', SAVE_DIR)